In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [4]:
target_files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

docs_by_name = {doc["filename"]: doc for doc in documents}
first_3 = [docs_by_name[name] for name in target_files]

In [5]:
d = documents[0]
print(type(d))
print(vars(d) if hasattr(d, "__dict__") else d)

<class 'dict'>
{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [6]:
import evaluation_utils
print([name for name in dir(evaluation_utils) if not name.startswith("_")])

['RAGBase', 'RAGWithUsage', 'calc_price', 'calc_total_price', 'llm_structured', 'llm_structured_retry', 'map_progress', 'time', 'tqdm']


In [7]:
import json
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import llm_structured

load_dotenv()
client = OpenAI()

In [8]:
class Questions(BaseModel):
    questions: list[str]

In [9]:
input_token_counts = []

for doc in first_3:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"],
    })

    parsed, usage = llm_structured(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="gpt-5.4-mini",
    )

    input_token_counts.append(usage.input_tokens)

avg = sum(input_token_counts) / len(input_token_counts)
avg

1354.0

In [14]:
import pandas as pd

df_gt = pd.read_csv("../data/ground-truth.csv")
ground_truth = df_gt.to_dict(orient="records")

In [15]:
from gitsource import chunk_documents
from minsearch import Index

chunks = chunk_documents(documents, size=2000, step=1000)  # 295 chunks

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [16]:
q = ground_truth[0]["question"]
results = text_search(q)
results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [20]:
from embedder import Embedder
from minsearch import VectorSearch

embed = Embedder()

X = embed.encode_batch([c["content"] for c in chunks])

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=num_results)

In [21]:
q = ground_truth[0]["question"]
results = vector_search(q)
results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [22]:
def compute_relevance(search_function, ground_truth):
    relevance_total = []
    for rec in ground_truth:
        results = search_function(rec["question"])
        relevance = [d["filename"] == rec["filename"] for d in results]
        relevance_total.append(relevance)
    return relevance_total

def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt = cnt + 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank]:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance_total)

def evaluate(search_function, ground_truth):
    relevance_total = compute_relevance(search_function, ground_truth)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [23]:
evaluate(text_search, ground_truth)

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [24]:
evaluate(vector_search, ground_truth)

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [25]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [26]:
for k in [1, 50, 100, 200]:
    search_fn = lambda q, k=k: hybrid_search(q, k=k)
    metrics = evaluate(search_fn, ground_truth)
    print(k, metrics)

1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}
50 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
100 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
200 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
